# Charts and binary coordinates

A **chart** is a coordinate map. Per-axis charts send a physical offset δ to a prior-normal `z`. A **physical binary chart** changes which names the sampler sees: ELL1-like `EPS1/EPS2/TASC` on a DDH engine that still uses `ECC/OM/T0`.

Theory lives in the package README. This notebook only shows the API.

AEI-DR2 combined J1022+1001 (DDH). This pulsar’s PX is negative, so expand at the prior center (not the par-file value). Secular dots and `STIG>1` keep the absorbed DDH (`fw10`) chart off.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

from pathlib import Path
from metapulsar import create_metapulsar
from nltiming import TimingSpec, TimingExpansionSpec
from nltiming.sampling import numpyro

numpyro.ensure_x64()

DATA = Path("..") / "data" / "J1022+1001"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1022+1001.par",
        "tim": DATA / "J1022+1001.tim",
        "timing_package": "tempo2",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
print(pulsar.name, len(pulsar.toas))


## Default `binary_chart="auto"`

Read `prior_chart` (δ→z) and `physical_chart` (engine vs sampling names). `binary_chart="auto"` may engage Kepler↔Laplace (`EPS1/EPS2/TASC`) or stay on engine `ECC/OM/T0` — print the manifest either way.


In [ ]:
spec = TimingSpec(
    engines="jug", binary_chart="auto", name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing = spec.for_pulsar(pulsar)
print(timing.plan)
print(timing.sampled)
print(timing.binary_chart_manifest())
for d in timing.chart_summary():
    print(d["name"], d["disposition"], d["prior_chart"], d["physical_chart"], d["engine_name"])


## Activate Kepler↔Laplace

Do not pass `binary_chart="on"` (it raises if a guard fails). Keep `EPS1` strictly positive so the prior box excludes the eccentricity origin.


In [ ]:
from nltiming.priors import uniform
from nltiming import laplace_from_kepler, kepler_from_laplace

spec_kl = TimingSpec(
    engines="jug",
    binary_chart="auto",
    priors={"EPS1": uniform(1e-6, 3e-4), "EPS2": uniform(-1e-4, 1e-4)},
    name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing_kl = spec_kl.for_pulsar(pulsar)
print(timing_kl.sampled)
print(timing_kl.delay_keys)
print(timing_kl.binary_chart_manifest())
for d in timing_kl.chart_summary():
    print(d["name"], d["prior_chart"], d["physical_chart"], d["engine_name"])

print(laplace_from_kepler(9.73e-5, 97.69, 50246.72, 7.805))
print(kepler_from_laplace(*laplace_from_kepler(9.73e-5, 97.69, 50246.72, 7.805), 7.805))


If the manifest still shows a skip reason (seam + `OMDOT`/`PBDOT`), stop here — do not tune priors in a loop.

## Enterprise parameter names

Identity static layer: one scalar `z` per sampled axis. Chart on → `..._timing_EPS1` instead of `..._timing_ECC`.


In [ ]:
from enterprise.signals import parameter, signal_base, white_signals

white = white_signals.MeasurementNoise(efac=parameter.Constant(1.0))
print(signal_base.PTA([(white + spec.enterprise_signal())(pulsar)]).param_names)
print(signal_base.PTA([(white + spec_kl.enterprise_signal())(pulsar)]).param_names)


In [ ]:
from nltiming import derived_kepler_columns
# after any posterior dict in sampling-frame names:
# derived_kepler_columns(post, timing_kl.binary_chart_manifest())


`RunResults.posterior()` / `derived_kepler_columns` map sampling-frame draws back to engine `ECC/OM/T0` when the chart is on.
